## A. Tworzenie projektu i magazynu danych.

### A.I Tworzenie i konfigurowanie projektu.

In [ ]:
# Projekt: Amundsen.
# Wersja: 1.0.1.

# Konfiguracja projektu z importowaniem wymaganych bibliotek.
import os
import shutil
from pyspark.sql import *
from pyspark.sql.types import *
from delta import *
from pathlib import Path

# JAVA_HOME jest wymagane dla prawidłowego importu biblioteki pyspark.
if "JAVA_HOME" not in os.environ:
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"

builder = (
    SparkSession
        .builder
        .appName("Amundsen")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

# Utworzenie sesji.
session = configure_spark_with_delta_pip(builder).getOrCreate()

# Konfiguracja dodatkowych parametrów sesji po jej utworzeniu.
session.conf.set("spark.sql.shuffle.partitions", "20")

### A.II Tworzenie bazy danych.

In [ ]:
# Wyświetlenie listy baz danych zarejestrowanych w katalogu.
session.catalog.listDatabases()

In [ ]:
# Fizyczne usunięcie magazynu danych (ang. datastore) tj. plików ze wskazanej lokalizacji.
warehouse_path = Path(session.conf.get("spark.sql.warehouse.dir")).resolve()

if warehouse_path.exists():
    shutil.rmtree(warehouse_path)

In [ ]:
# Warunkowe usunięcie bazy danych earth.
session.sql("DROP DATABASE IF EXISTS earth CASCADE")

# Warunkowe utworzenie bazy danych earth.
session.sql("CREATE DATABASE IF NOT EXISTS earth")

# Wskazanie bieżącej bazy danych w ramach sesji.
session.sql("USE earth")
session.catalog.currentDatabase()

## B. Definiowanie tabeli w formacie Delta na podstawie danych.

### B.I Definiowanie tabeli w formacie Delta z wykorzystaniem ramki danych DF.

In [ ]:
# Utworzenie kolekcji danych.
data = [
    (1, "Afryka"),
    (2, "Ameryka Południowa"),
    (3, "Ameryka Północna"),
    (4, "Antarktyda"),
    (5, "Australia"),
    (6, "Azja"),
    (7, "Europa")
]

# Utworzenie i wyświetlenie ramki danych DF (ang. Data Frame).
df = session.createDataFrame(
    data,
    ["continent_id", "continent"]
)
df.show()

In [ ]:
# Utworzenie tabeli typu Delta
df.write.format("delta").mode("overwrite").saveAsTable("continent")

In [ ]:
# Wyświetlenie danych tabeli via DataFrame API (PySpark).
session.table("continent").orderBy("continent_id").show()

#### <span style="color: blue;">PYTANIE: Jaką lokalizację oraz strukturę mają pliki w których przechowywane są dane tabeli continent? Sprawdź w lokalizacji bazy danych.</span>

In [ ]:
# Wyświetlenie listy baz danych zarejestrowanych w katalogu wraz z LOKALIZACJĄ.
session.catalog.listDatabases()

### B.II Definiowanie tabeli w formacie Delta z wykorzystaniem kolekcji RDD oraz ramki danych DF.

In [ ]:
# Utworzenie kolekcji danych.
data = [
    (1, "Lambert–Fisher Glacier", 97000),
    (2, "Agassiz Ice Cap", 21000),
    (3, "Severny Island Ice Cap", 20000),
    (4, "Prince of Wales Icefield", 19000),
    (5, "Devon Ice Cap", 14000),
    (6, "Southern Patagonian Ice Field", 13000),
    (7, "Austfonna", 8000),
    (8, "Vatnajökull", 7700)
]

# Utworzenie rozproszonej kolekcji RDD (ang. Resilient Distributed Datasets).
rdd = session.sparkContext.parallelize(data, 20)

# Utworzenie i wyświetlenie ramki danych DF (ang. Data Frame).
columns = ["glacier_id", "glacier", "area"]
df = rdd.toDF(columns)
df.show()

In [ ]:
# Utworzenie tabeli typu Delta.
df.write.format("delta").mode("overwrite").saveAsTable("glacier")

In [ ]:
# Wyświetlenie danych tabeli via DataFrame API (PySpark).
session.table("glacier").orderBy("area", ascending=False).show()

### B.III Definiowanie tabeli w formacie Delta na podstawie danych tabeli w formacie Parquet.

In [ ]:
# Utworzenie kolekcji danych.
data = [
    (1, "Ocean Atlantycki"),
    (2, "Ocean Arktyczny"),
    (3, "Ocean Indyjski"),
    (4, "Ocean Południowy"),
    (5, "Ocean Spokojny")
]

# Utworzenie rozproszonej kolekcji RDD (ang. Resilient Distributed Datasets).
rdd = session.sparkContext.parallelize(data, 20)

# Utworzenie i wyświetlenie ramki danych DF (ang. Data Frame).
columns = ["ocean_id", "ocean"]
df = rdd.toDF(columns)
df.show()

In [ ]:
# Sprawdzenie lokalizacji Current Working Directory oraz Warehouse Directory.
print("Current directory:", os.getcwd())
print("Spark warehouse:", session.conf.get("spark.sql.warehouse.dir"))

In [ ]:
# Utworzenie kolekcji danych w formacie Parquet.
parquet_data_path = (
    session.conf.get("spark.sql.warehouse.dir")
    + "/"
    + session.catalog.currentDatabase()
    + ".db/"
    + "ocean"
)

df.write.format("parquet").mode("overwrite").save(parquet_data_path)

In [ ]:
# Sprawdzenie formatu tabeli ocean.
DeltaTable.isDeltaTable(session, parquet_data_path)

In [ ]:
# Utworzenie tabeli Delta poprzez konwersję z formatu Parquet.
delta_table = DeltaTable.convertToDelta(
    session,
    f"parquet.`{parquet_data_path}`"
)

In [ ]:
# Ponowne sprawdzenie formatu tabeli ocean.
DeltaTable.isDeltaTable(session, parquet_data_path)

In [ ]:
# Wyświetlenie informacji o tabeli POZIOM LOGICZNY {CATALOG}.
session.sql(
    """
    SHOW TABLE EXTENDED IN earth LIKE 'ocean'
    """
).show(truncate=False)

#### <span style="color: blue;">PYTANIE: Czy jakiekolwiek informacje o tabeli są wyświetlane?</span>

In [ ]:
# Rejestracja danych w formacie Delta w katalogu tj. utworzenie tabeli w katalogu.
session.sql(
    f"""
    CREATE TABLE IF NOT EXISTS ocean
    LOCATION '{parquet_data_path}'
    """
)

In [ ]:
# Wyświetlenie informacji o tabeli POZIOM LOGICZNY {CATALOG}.
session.sql(
    """
    SHOW TABLE EXTENDED IN earth LIKE 'ocean'
    """
).show(truncate=False)

In [ ]:
# Wyświetlenie informacji o tabeli POZIOM FIZYCZNY {DATASTORE}.
session.sql(
    """
    DESCRIBE DETAIL ocean
    """
).show(truncate=False)

In [ ]:
# Wyświetlenie danych tabeli via DataFrame API (PySpark).
session.table("ocean").orderBy("ocean").show()

### B.IV Definiowanie tabeli w formacie Delta na podstawie danych z pliku CSV.

#### <span style="color: green;">WSKAZÓWKA: Do utworzenia ramki danych, konieczne jest umieszczenie w lokalizacji:</span>
<span style="color: green;">CURRENT_WORKING_DIRECTORY/data/volcano</span>
#### <span style="color: green;">plików volcano_01.csv, volcano_02.csv</span>

In [ ]:
# Utworzenie i wyświetlenie ramki danych.

data_path = (
    Path.cwd()
    / "data"
    / "volcano"
)

df = session.read \
    .option("header", True) \
    .csv(data_path.resolve().as_uri())
df.show()

In [ ]:
# Utworzenie tabeli w formacie Delta.
df.write.format("delta").mode("overwrite").saveAsTable("volcano")

In [ ]:
# Wyświetlenie informacji o tabeli POZIOM LOGICZNY {CATALOG}.
session.sql(
    """
    SHOW TABLE EXTENDED IN earth LIKE 'volcano'
    """
).show(truncate=False)

In [ ]:
# Wyświetlenie danych tabeli via DataFrame API (PySpark).
session.table("volcano").show()

#### <span style="color: red;">B. ZADANIE I</span>
a) Utwórz tabelę w formacie Delta zarejestrowaną w katalogu pod nazwą lake na podstawie danych z 3 plików w lokalizacji /data/lake/.<br/>
b) Wyświetl dane tabeli.

#### <span style="color: blue;">PYTANIE: Jaka jest liczba wierszy w tabeli lake? Czy dane się powtarzają?</span>

## C. Definiowanie tabeli Delta na podstawie jawnego schematu.

### C.I Definiowanie tabeli Delta z wykorzystaniem SQL.

In [ ]:
# Utworzenie tabeli typu Delta z wykorzystaniem SQL.
session.sql(
    """
    CREATE OR REPLACE TABLE peak (
        peak_id integer,
        peak string,
        height integer
    ) USING delta
    COMMENT 'Essential information on mountain peaks.'
    """
)

In [ ]:
# Wyświetlenie informacji o tabeli POZIOM LOGICZNY {CATALOG}.
session.sql(
    """
    SHOW TABLE EXTENDED IN earth LIKE 'peak'
    """
).show(truncate=False)

In [ ]:
# Wyświetlenie wybranych informacji o tabeli POZIOM FIZYCZNY {DATASTORE}.
session.sql(
    """
    DESCRIBE DETAIL peak
    """
).show(truncate=False)

### C.II Definiowanie tabeli Delta z wykorzystaniem StructType.

In [ ]:
# Utworzenie tabeli w formacie Delta z wykorzystaniem StructType.
river_schema = StructType([
    StructField("river_id", IntegerType(), False),
    StructField("river", StringType(), False),
    StructField("length", IntegerType(), True)
])
data = NullType
df = session.createDataFrame([], schema=river_schema)

# Utworzenie tabeli w formacie Delta.
df.write.format("delta").mode("overwrite").saveAsTable("river")

In [ ]:
# Wyświetlenie wybranych informacji o tabeli POZIOM FIZYCZNY {DATASTORE}.
session.sql(
    """
    DESCRIBE DETAIL river
    """
).show(truncate=False)

### C.III Definiowanie tabeli Delta z wykorzystaniem DeltaTable API.

In [ ]:
# Utworzenie tabeli w formacie Delta z wykorzystaniem DeltaTable API
(
    DeltaTable
    .createOrReplace(session)
    .tableName("country")
    .addColumn("country_id", dataType="integer", nullable=False)
    .addColumn("country", dataType="string", nullable=False)
    .addColumn("continent_id", dataType=IntegerType())
    .partitionedBy("continent_id")
    .execute()
)

In [ ]:
# Wyświetlenie wybranych informacji o tabeli POZIOM FIZYCZNY {DATASTORE}.
session.sql(
    """
    DESCRIBE DETAIL country
    """
).show(truncate=False)

#### <span style="color: red;">C. ZADANIE I</span>
a) Uzupełnij tabelę <b>peak</b> danymi.<br/>
b) Wyświetl dane tabeli.
| id | peak | height |
|-----------|-----------|-----------|
| 1     | Aconcagua   | 6961 |
| 2       | Denali  | 6190 |
| 3       | K2  | 8611 |
| 4       | Lhotse  | 8516 |
| 5       | Vinson  | 4892 |


#### <span style="color: red;">C. ZADANIE II</span>
a) Uzupełnij tabelę <b>country</b> danymi.<br/>
b) Wyświetl dane tabeli.
| id | country | continent_id |
|-----------|-----------|-----------|
| 1     | Argentyna   | 2 |
| 2       | Chiny  | 6 |
| 3       | Indie  | 6 |
| 4       | Nepal  | 6 |
| 5       | Pakistan  | 6 |
| 6       | Polska  | 7 |
| 7       | Stany Zjednoczone  | 3 |
| 8       | Tadżykistan  | 6 |

#### <span style="color: red;">C. ZADANIE III</span>
a) Uzupełnij tabelę <b>peak_country</b> danymi.<br/>
b) Wyświetl dane tabeli.
| peak_id | country_id |
|-----------|-----------|
| 1     | 1 |
| 2       | 7 |
| 3       | 2 |
| 3       | 5 |
| 4       | 2 |
| 4      | 4 |
| 5       | NULL |

#### <span style="color: red;">C. ZADANIE IV</span>
Zdefiniuj ograniczenia kluczy podstawowych oraz obcych dla tabel <b>peak</b>, <b>country</b> oraz <b>peak_country</b>.

#### <span style="color: blue;">PYTANIE: Czy zdefiniowanie ograniczeń kluczy jest możliwe? Jakie ograniczenia są możliwe do zdefiniowania?</span>

### D. Podstawowe operacje na tabelach

### D.I Wyświetlanie informacji o tabeli POZIOM FIZYCZNY {DATASTORE}.

In [ ]:
# Wyświetlenie pełnych informacji o tabeli POZIOM FIZYCZNY {DATASTORE}.
session.sql(
    """
    DESCRIBE DETAIL peak
    """
).show(truncate=False)

In [ ]:
# Wyświetlenie wybranych informacji o tabeli POZIOM FIZYCZNY {DATASTORE}.
session.sql(
    """
    DESCRIBE DETAIL peak
    """
).select("format").show(truncate=False)

### D.II Wyświetlanie informacji o tabeli POZIOM LOGICZNY {CATALOG}.

In [ ]:
# Wyświetlanie informacji o tabeli/tabelach POZIOM LOGICZNY {CATALOG}.
session.sql(
    """
    SHOW TABLE EXTENDED IN earth LIKE 'c*'
    """
).show(truncate=False)

### D.III Wyświetlanie danych tabeli

In [ ]:
# Wyświetlenie danych tabeli via DataFrame API (PySpark).
session.table("continent").orderBy("continent_id").show()

In [ ]:
# Wyświetlenie danych tabeli via Spark SQL (PySpark).
session.sql(
    """
    SELECT * FROM continent ORDER BY 1
    """
).show()

### D.IV Optymalizacja tabeli

In [ ]:
# Optymalizacja tabeli.
dt_continent = DeltaTable.forName(session, "continent")
dt_continent.optimize().executeCompaction()

#### <span style="color: blue;">PYTANIE: Czym jest dt_continent?</span>

#### Wskazówka. Wykonaj analizę n/w przypadków.

In [ ]:
# (1-A-a)
dt_continent = DeltaTable.forPath(
    session,
    session.conf.get("spark.sql.warehouse.dir") + "/" + session.catalog.currentDatabase() + ".db/" + "continent"
)
# (1-A-b)
dt_continent = DeltaTable.forName(session, "continent")
# (1-A-c) Metadane tabeli Delta
dt_continent.detail().show(truncate=False)

In [ ]:
# (2-A) Metadane tabeli
print(session.catalog.getTable("continent"))
# (1-B-a)
df_continent = session.table("continent")
# (1-B-b) Schemat kolekcji DataFrame
df_continent.printSchema()
# (1-B-c) Charakterystyka danych kolekcji DataFrame
df_continent.describe().show()

In [ ]:
# (2-A) Pełne informacje o tabeli
session.sql(
    """
    DESCRIBE EXTENDED continent
    """
).show(truncate=False)
# (2-B) Dane tabeli bezpośrednio via katalog MetaStore.
session.sql(
    """
    SELECT * FROM continent WHERE FALSE ORDER BY 1
    """
).show()

In [ ]:
# (3)
session.sql("DESCRIBE DETAIL continent").first()["location"]